```python
# ---------------------------------------------------------
# 1. 라이브러리 및 데이터 불러오기
# ---------------------------------------------------------

import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
```

```python
# ---------------------------------------------------------
# 2. EDA (탐색적 데이터 분석)
# ---------------------------------------------------------

# 데이터 미리보기
print(train.head())
print(test.head())

# 행, 컬럼 개수
print(train.shape)
print(test.shape)

# 컬럼 정보 및 자료형
train.info()
test.info()

# 기초 통계량
print(train.describe())                    # 수치형
print(train.describe(include='object'))    # 범주형
print(train.describe(include='all'))       # 전체

# 결측치 확인
print(train.isnull().sum())
print(test.isnull().sum())

# 타깃 분포 확인 (분류)
print(train['income'].value_counts())
print(train['income'].value_counts(normalize=True))

# 컬럼명 확인
print(train.columns)
print(test.columns)
```

```python
# ---------------------------------------------------------
# 3. 타깃 분리 및 ID 처리
# ---------------------------------------------------------

# 타깃 분리
target = 'income'  # 문제에서 예측하라는 컬럼명
y_train = train.pop(target)

# 제출용 ID 보관
test_id = test['id'].copy()

# ID 컬럼 삭제
train = train.drop(columns=['id'], axis= 1)
test = test.drop(columns=['id'], axis= 1)

print(train.shape, test.shape, y_train.shape)
```

```python
# ---------------------------------------------------------
# 4. 결측치 처리 (결측치 있으면 필수)
# ---------------------------------------------------------

# ------------ 결측치 확인 ------------
print(train.isnull().sum())           # 컬럼별 결측치 개수
print(train.isnull().sum().sum())     # 전체 결측치 개수
print(train.isnull().mean() * 100)    # 결측치 비율(%)

# ------------ 결측치 삭제 ------------
# [행 삭제]
df = train.dropna()                           # NaN이 하나라도 있는 행 삭제
df = train.dropna(how='all')                  # 모든 값이 NaN인 행 삭제

# 특정 컬럼 기준 [행] 삭제
df = train.dropna(subset=['age', 'workclass'])
df = train.dropna(subset=['age', 'workclass'], how='any')  # 하나라도 NaN
df = train.dropna(subset=['age', 'workclass'], how='all')  # 둘 다 NaN

# [컬럼 삭제]
df = train.dropna(axis=1)             # NaN이 하나라도 있는 컬럼 삭제
df = train.dropna(axis=1, how='all')  # 모든 값이 NaN인 컬럼 삭제

# 지정한 [컬럼] 삭제 
df = train.drop(columns=['age', 'workclass'], axis=1)

# ------------ 결측치 대체 ------------
# [수치형]
df['age'] = df['age'].fillna(df['age'].mean())    # 평균
df['age'] = df['age'].fillna(df['age'].median())  # 중앙값
df['age'] = df['age'].fillna(0)                   # 0

# [범주형]
m = df['workclass'].mode()[0]
df['workclass'] = df['workclass'].fillna(m)          # 최빈값
df['workclass'] = df['workclass'].fillna('Unknown')  # 문자열

# [전체]
df = df.fillna(0)

##### 테스트 데이터
test['workclass'] = test['workclass'].fillna(train['workclass'].mode()[0])
test['country'] = test['country'].fillna(train['country'].mode()[0])
test['occupation'] = test['occupation'].fillna('X')

# ------------ train/test 결측치 대체 (기본 추천) ------------
# 컬럼 구분
num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(include=['object', 'category']).columns

# 수치형: 중앙값
for col in num_cols:
    value = train[col].median()
    train[col] = train[col].fillna(value)
    test[col] = test[col].fillna(value)

# 범주형: 문자열
for col in cat_cols:
    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')
```

```python
# ---------------------------------------------------------
# 5. 이상치 처리 (선택)
# ---------------------------------------------------------
# ------------ 조건으로 이상치 처리 ------------
# 특정 범위 행만 선택
df = train[train['age'] >= 0]
df = train[train['age'] <= 100]

# 여러 조건
df = train[(train['age'] >= 0) & (train['age'] <= 100)]

# 특정 값 제거
df = train[train['age'] != -999]

# 특정 값 대체
train['age'] = train['age'].replace(-999, 0)

# ------------ IQR 방식 ------------
Q1 = train['age'].quantile(0.25)
Q3 = train['age'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# 이상치 개수
print(((train['age'] < lower) | (train['age'] > upper)).sum())

# [방법 1] 이상치 행 삭제
df = train[(train['age'] >= lower) & (train['age'] <= upper)]

# [방법 2] 이상치만 조회
df = train[(train['age'] < lower) | (train['age'] > upper)]

# [방법 3] 경계값으로 대체
train['age'] = train['age'].clip(lower, upper)
test['age'] = test['age'].clip(lower, upper)
```

```python
# ---------------------------------------------------------
# 6. 인코딩 (범주형 데이터 있으면 필수)
# ---------------------------------------------------------
# ------------ 컬럼 구분 ------------
# 범주형 컬럼
cat_cols = train.select_dtypes(include='object').columns

# 수치형 컬럼
num_cols = train.select_dtypes(include='number').columns

# 쉬운 버전
cols = ['product', 'location']

# ------------ 원핫인코딩 ------------
print(train.shape, test.shape)

data = pd.concat([train, test], axis=0)
data_oh = pd.get_dummies(data)
 
train = data_oh.iloc[:len(train)].copy()
test = data_oh.iloc[len(train):].copy()

print(train.shape, test.shape)

# ------------ 레이블인코딩 ------------
from sklearn.preprocessing import LabelEncoder

for col in cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
```

```python
# ---------------------------------------------------------
# 7. 스케일링 (선택)
# ---------------------------------------------------------

# 랜덤포레스트는 스케일링이 필요없다.
cols = train.select_dtypes(include='object').columns
cols = ['age', 'salary']

def get_data():
	train_copy = train.copy()
	test_copy = test.copy()
	return train_copy, test_copy
	
train_copy, test_copy = get_data()

# ------------ MinMaxScaler: 학습 데이터 기준 0~1 ------------
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# ------------ StandardScaler: 평균 0, 표준편차 1 ------------
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# ------------ RobustScaler: 중앙값과 IQR ------------
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()

# 공통 
train_copy[cols] = scaler.fit_transform(train_copy[cols])
test_copy[cols] = scaler.transform(test_copy[cols])
```

```python
# ---------------------------------------------------------
# 8. 검증 데이터 분할
# ---------------------------------------------------------
print(train.shape, y_train.shape, test.shape)

from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(train, y_train, test_size=0.2, random_state=42, stratify=y_train)  
# stratify=y_train 분류에서만 사용
											

print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)
```

```python
# ---------------------------------------------------------
# 9-1. 머신러닝 학습 및 평가 - 분류
# RandomForest / LightGBM / GradientBoosting
# ---------------------------------------------------------
# 학습(fit) -> 예측(predict) -> 평가
# Accuracy, F1 : predict()
# ROC-AUC      : predict_proba()[:, 1]

# ------------ 모델 선택 (하나만 사용) ------------
# RandomForest
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)

# LightGBM (설치된 환경에서 사용)
import lightgbm as lgb
model = lgb.LGBMClassifier(random_state=42, verbose=-1)

# GradientBoosting
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(random_state=42)

# ------------ 학습 ------------
model.fit(X_tr, y_tr) 

# ------------ 예측 ------------
pred = model.predict(X_val)              # Accuracy, F1
proba = model.predict_proba(X_val)[:, 1]  # ROC-AUC

print(pred[:10])
print(proba[:10])
print(model.classes_)

# ------------ 평가 ------------
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 정확도
print(accuracy_score(y_val, pred))

# F1 (양성 클래스가 1일 때)
print(f1_score(y_val, pred))

# F1 (양성 클래스가 문자열일 때)
# print(f1_score(y_val, pred, pos_label='>50K'))

# ROC-AUC (이진 분류, 양성 클래스가 classes_[1])
print(roc_auc_score(y_val, proba))

# ------------ 모델 비교 ------------
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

models = {
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, proba)

    print(name, score)
    
    
# ------------ 핵심 암기 ------------
model = RandomForestClassifier(random_state=42)
model.fit(X_tr, y_tr)

pred = model.predict(X_val)
proba = model.predict_proba(X_val)[:, 1]

print(roc_auc_score(y_val, proba))
```

```python
# ---------------------------------------------------------
# 9-2. 머신러닝 학습 및 평가 - 회귀
# RandomForest / LightGBM / GradientBoosting
# ---------------------------------------------------------

# ------------ 모델 선택 (하나만 사용) ------------
# 선형회귀
from sklearn.linear_model import LinearRegression
model = LinearRegression()

# RandomForest
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

# LightGBM (설치된 환경에서 사용)
import lightgbm as lgb
model = lgb.LGBMRegressor(random_state=42, verbose=-1)

# GradientBoosting
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(random_state=42)

# ------------ 학습 ------------
model.fit(X_tr, y_tr)

# ------------ 예측 ------------
pred = model.predict(X_val)

# ------------ 평가 ------------
import numpy as np
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

print(np.sqrt(mean_squared_error(y_val, pred))) # RMSE
print(root_mean_squared_error(y_val, pred)) # RMSE
print(mean_squared_error(y_val, pred)) # MSE
print(mean_absolute_error(y_val, pred)) # MAE
print(r2_score(y_val, pred)) # R2

# RMSLE (실제값과 예측값이 0 이상이어야 함)
print(np.sqrt(mean_squared_log_error(
    y_val, np.maximum(pred, 0)
)))

# ------------ 모델 비교 ------------
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

models = {
    'RandomForest': RandomForestRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42)
}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    score = np.sqrt(mean_squared_error(y_val, pred))

    print(name, score)
    
    
# ------------ 핵심 암기 ------------
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model = RandomForestRegressor(random_state=42)
model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print(np.sqrt(mean_squared_error(y_val, pred)))  # RMSE
```

```python
# ---------------------------------------------------------
# 10. 예측 및 결과 파일 생성
# ---------------------------------------------------------

# ------------ 분류 — 클래스 제출 ------------
pred = model.predict(test)

# ------------ 분류 — 확률 제출 (AUC) ------------
# 양성 클래스가 1인 경우
idx = list(model.classes_).index(1)
pred = model.predict_proba(test)[:, idx]

# ------------ 회귀 — 수치 제출 ------------
pred = model.predict(test)

# ------------ 제출 파일 생성 및 확인 ------------
# 파일 생성
result = pd.DataFrame({
    'id': test_id,
    'price': pred
})
result.to_csv('result.csv', index=False)

# 파일 확인
result = pd.read_csv('result.csv')
print(result.head())
print(result.shape)
print(result.isnull().sum())
print(result.columns)
```

### 문제 유형별로 바꿀 부분

| 문제 | 모델 | 예측 |
| --- | --- | --- |
| 이진분류·Accuracy/F1 | `RandomForestClassifier` | `predict()` |
| 이진분류·AUC | `RandomForestClassifier` | `predict_proba()[:, 1]` |
| 다중분류·Macro F1 | `RandomForestClassifier` | `predict()` |
| 회귀·RMSE/MAE | `RandomForestRegressor` | `predict()` |
| 회귀·RMSLE | `RandomForestRegressor` | 음수 예측값 처리에 주의 |